# Advanced RAG · Agentic and Graph Patterns (runnable on Bedrock)

Direct Amazon Bedrock, no LiteLLM. The self-correcting loops are written as plain, transparent Python so you can watch every decision. In production these map onto LangGraph nodes, but here nothing is hidden behind a framework.

**What is inside**

| Pattern | Idea |
|---|---|
| Adaptive RAG | route by query complexity: none / single / multi |
| Self-RAG | reflect: decide to retrieve, grade passages, check support |
| CRAG | grade the retrieval, then correct it (refine or web search) |
| Agentic RAG | retrieval as a tool the model chooses to call |
| Mini GraphRAG | entity graph, communities, summaries, global and local search |

**Run it (VS Code)**
1. `python -m venv .venv`, activate it, select `.venv` as the kernel.
2. `pip install -U boto3 numpy networkx`
3. `aws configure` (or set `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` / `AWS_REGION`).
4. Run cells top to bottom.

**Run it (Google Colab)**
1. `!pip install -q boto3 numpy networkx`
2. Set creds via Colab secrets or env vars.
3. Run cells top to bottom.

Model: `us.anthropic.claude-haiku-4-5-20251001-v1:0` (the `us.` inference-profile prefix is required). Embeddings: `amazon.titan-embed-text-v2:0`.

In [ ]:
# If running fresh, uncomment:
# %pip install -U boto3 numpy networkx
import boto3, json, re
import numpy as np

In [ ]:
REGION = "us-east-1"
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. inference profile required
EMBED_MODEL = "amazon.titan-embed-text-v2:0"

brt = boto3.client("bedrock-runtime", region_name=REGION)

try:
    ident = boto3.client("sts", region_name=REGION).get_caller_identity()
    print("AWS creds OK. Account:", ident["Account"])
except Exception as e:
    print("No AWS creds detected. The graph build (networkx) still runs; Bedrock")
    print("cells work once creds are set. Detail:", str(e)[:160])

In [ ]:
def chat(prompt, system=None, max_tokens=500, temperature=0.0):
    kwargs = dict(
        modelId=MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
    )
    if system:
        kwargs["system"] = [{"text": system}]
    resp = brt.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"].strip()

def embed(text):
    resp = brt.invoke_model(modelId=EMBED_MODEL, body=json.dumps({"inputText": text}))
    return np.array(json.loads(resp["body"].read())["embedding"], dtype=float)

def embed_many(texts):
    return np.vstack([embed(t) for t in texts])

def cosine_scores(matrix, q):
    mn = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
    qn = q / (np.linalg.norm(q) + 1e-9)
    return mn @ qn

## A small corpus and a plain retriever

The TravelMind policy base. Note there is no "pet in cabin" rule on purpose, so the CRAG correction path has something to trigger on.

In [ ]:
CORPUS = {
  "refunds":    "TravelMind refunds. A cancelled flight is fully refundable to the original payment method within 7 to 10 business days.",
  "change_fee": "Change fees by tier. Basic and Silver members pay a change fee. Gold and Platinum members have the change fee waived on eligible fares.",
  "tiers":      "Loyalty tiers: Basic, Silver, Gold, Platinum, based on annual miles. Gold unlocks waived change fees, priority boarding, and extra baggage.",
  "pnr":        "A PNR is a six character code identifying a booking, for example JX48Q2. Use the PNR and last name to retrieve a reservation.",
  "checkin":    "Online check in opens 48 hours before departure and closes 60 minutes before for domestic flights.",
  "baggage":    "Basic fares include one cabin bag up to 7 kg. Gold and above receive one extra checked bag.",
  "seat":       "Seats can be changed free of charge until check in, subject to availability.",
}
IDS = list(CORPUS)
TEXTS = [CORPUS[k] for k in IDS]
MATRIX = embed_many(TEXTS)

def retrieve(query, k=4):
    sims = cosine_scores(MATRIX, embed(query))
    return [(IDS[i], CORPUS[IDS[i]]) for i in np.argsort(-sims)[:k]]

print("index:", MATRIX.shape)

## Pattern 1 · Adaptive RAG (route by complexity)

Match effort to difficulty. Classify the query, then take the cheapest path that can answer it: no retrieval, a single lookup, or a multi-hop decomposition.

```mermaid
flowchart TB
    Q["Query"] --> C["Classifier"]
    C -->|none| A["Answer directly"]
    C -->|single| B["One retrieval"]
    C -->|multi| D["Decompose, retrieve each"]
```

In [ ]:
def classify_complexity(q):
    r = chat(
        "Classify this question for a retrieval system as one of: none, single, multi. "
        "'none' = general knowledge, 'single' = one lookup, 'multi' = needs several. "
        f"Reply with one word.\n\nQuestion: {q}",
        max_tokens=10,
    ).lower()
    for lab in ("multi", "single", "none"):
        if lab in r:
            return lab
    return "single"

def adaptive_rag(q):
    route = classify_complexity(q)
    if route == "none":
        return {"route": route, "answer": chat(q, max_tokens=200)}
    if route == "single":
        docs = retrieve(q, k=4)
        ctx = "\n".join(f"[{c}] {t}" for c, t in docs)
        ans = chat(f"Answer from the context, cite [id].\n\n{ctx}\n\nQuestion: {q}", max_tokens=250)
        return {"route": route, "answer": ans}
    subs = [s for s in chat(f"Break this into 2 to 3 sub-questions, one per line.\n\n{q}",
                            max_tokens=120).splitlines() if s.strip()]
    notes = []
    for s in subs:
        docs = retrieve(s, k=3)
        notes.append(f"Q: {s}\n" + "\n".join(t for _, t in docs))
    ans = chat("Answer the original question using these notes.\n\n" +
               "\n\n".join(notes) + f"\n\nOriginal: {q}", max_tokens=300)
    return {"route": route, "answer": ans}

print(adaptive_rag("Is a cancelled flight refundable?"))

## Pattern 2 · Self-RAG (reflect as you go)

Written as a transparent state machine. Decide whether to retrieve, grade each passage for relevance, generate, then check the answer is supported. Retry on failure, with a hard cap so the loop always ends.

```mermaid
stateDiagram-v2
    [*] --> Decide
    Decide --> Direct: no retrieve
    Decide --> Retrieve: retrieve
    Retrieve --> Grade
    Grade --> Generate
    Generate --> Reflect
    Reflect --> [*]: supported
    Reflect --> Retrieve: not supported (capped)
    Direct --> [*]
```

In [ ]:
def sr_decide(q):
    r = chat(f"Question: {q}\nCan you answer from general knowledge, or do you need to "
             "retrieve documents? Reply 'direct' or 'retrieve'.", max_tokens=10)
    return "direct" if "direct" in r.lower() else "retrieve"

def sr_grade(q, docs):
    kept = []
    for cid, text in docs:
        r = chat(f"Question: {q}\nPassage: {text}\nIs this passage relevant to the "
                 "question? Reply yes or no.", max_tokens=5)
        if r.lower().startswith("y"):
            kept.append((cid, text))
    return kept

def sr_generate(q, docs):
    ctx = "\n".join(f"[{c}] {t}" for c, t in docs)
    return chat(f"Answer only from the context and cite [id].\n\n{ctx}\n\nQuestion: {q}", max_tokens=250)

def sr_supported(ans, docs):
    ctx = "\n".join(t for _, t in docs)
    r = chat(f"Context: {ctx}\nAnswer: {ans}\nIs the answer fully supported by the "
             "context? Reply yes or no.", max_tokens=5)
    return r.lower().startswith("y")

def self_rag(q, max_attempts=2):
    if sr_decide(q) == "direct":
        return {"path": "direct", "answer": chat(q, max_tokens=200)}
    query, ans, attempt = q, None, 0
    while attempt < max_attempts:
        attempt += 1
        docs = retrieve(query, k=4)
        kept = sr_grade(q, docs) or docs           # fall back if grader drops all
        ans = sr_generate(q, kept)
        if sr_supported(ans, kept):
            return {"path": "retrieve", "attempts": attempt, "answer": ans}
        query = chat(f"Rewrite to retrieve better evidence: {q}", max_tokens=40)
    return {"path": "retrieve-capped", "attempts": attempt, "answer": ans}

print(self_rag("Do Gold members pay a change fee?"))

## Pattern 3 · CRAG (grade the retrieval, then correct)

A dedicated evaluator grades the retrieval as correct, ambiguous, or incorrect, then acts: refine the good strips, or fall back to an external source. The web search here is a stub that broadens retrieval. In production, swap in a real web tool (Tavily, or AgentCore Web Search).

```mermaid
flowchart TB
    R["Retrieve"] --> G{"Grade"}
    G -->|correct| RF["Refine strips"]
    G -->|ambiguous| BO["Refine + web"]
    G -->|incorrect| WB["Web search"]
    RF --> GN["Generate"]
    BO --> GN
    WB --> GN
```

In [ ]:
def crag_grade(q, docs):
    joined = "\n".join(t for _, t in docs)
    r = chat(f"Question: {q}\nRetrieved:\n{joined}\nGrade the retrieval for answering the "
             "question as one word: correct, ambiguous, or incorrect.", max_tokens=10).lower()
    for g in ("incorrect", "ambiguous", "correct"):
        if g in r:
            return g
    return "ambiguous"

def crag_refine(q, docs):
    joined = "\n".join(t for _, t in docs)
    return chat(f"From the text, extract only the parts relevant to the question.\n\n"
                f"Question: {q}\nText:\n{joined}", max_tokens=200)

def web_search_stub(q):
    # Production: a real web search tool. Here: broaden the internal retrieval.
    return "\n".join(t for _, t in retrieve(q, k=6))

def crag(q):
    docs = retrieve(q, k=4)
    grade = crag_grade(q, docs)
    if grade == "correct":
        context = crag_refine(q, docs)
    elif grade == "incorrect":
        context = web_search_stub(q)
    else:
        context = crag_refine(q, docs) + "\n" + web_search_stub(q)
    ans = chat(f"Answer only from the context. If it is not present, say you do not know.\n\n"
               f"Context:\n{context}\n\nQuestion: {q}", max_tokens=250)
    return {"grade": grade, "answer": ans}

print(crag("What is the refund window for a cancelled flight?"))
print(crag("Can I bring my cat in the cabin?"))   # not in corpus -> correction path

## Pattern 4 · Agentic RAG (retrieval as a tool)

Instead of hard-wiring retrieval, hand it to the model as a tool it may call, possibly several times, before answering. A simple text protocol keeps the loop transparent: the model replies with either `SEARCH: <query>` or `ANSWER: <text>`. A step cap prevents runaway loops.

In [ ]:
def agentic_rag(q, max_steps=3):
    notes = []
    for step in range(max_steps):
        context = "\n".join(notes) if notes else "(no lookups yet)"
        r = chat(
            "You answer TravelMind questions and may look up the knowledge base.\n"
            "Reply with exactly one line:\n"
            "  SEARCH: <query>   to look something up, or\n"
            "  ANSWER: <text>    to give the final answer with [id] citations.\n\n"
            f"Question: {q}\nNotes so far:\n{context}",
            max_tokens=120,
        )
        line = r.strip().splitlines()[0]
        if line.upper().startswith("SEARCH:"):
            query = line.split(":", 1)[1].strip()
            hits = retrieve(query, k=3)
            notes.append(f"Searched '{query}': " + "; ".join(f"[{c}] {t}" for c, t in hits))
        else:
            ans = line.split(":", 1)[1].strip() if ":" in line else line
            return {"steps": step + 1, "answer": ans}
    ans = chat("Answer with [id] citations using these notes.\n\n" + "\n".join(notes) +
               f"\n\nQuestion: {q}", max_tokens=200)
    return {"steps": max_steps, "answer": ans}

print(agentic_rag("As a Gold member, do I pay to change a flight, and how long for a refund?"))

## Pattern 5 · Mini GraphRAG

Vector RAG cannot answer "what connects these entities" or "what are the main themes". GraphRAG (Microsoft, 2024) builds an entity graph, detects communities, summarizes each, then answers global questions by map-reduce over summaries and local questions by traversing neighbors.

This is a toy version: an LLM extracts entity relationships, `networkx` builds the graph and finds communities (production uses the Leiden algorithm), and the model writes community summaries.

In [ ]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

GRAPH_DOCS = [
    "A storm at Bengaluru airport BLR delayed several flights.",
    "Flight JX48Q2 from BLR to Delhi was cancelled due to the storm.",
    "Passenger Rao was booked on flight JX48Q2.",
    "Rao holds TravelMind Gold tier status.",
    "TravelMind rebooked Rao onto flight JX51A8.",
    "TravelMind partners with PartnerBank on a co-branded credit card.",
    "PartnerBank cardholders earn double miles on TravelMind flights.",
    "The PartnerBank card waives the first checked bag fee.",
]

def extract_triples(text):
    out = chat("Extract entities and their relationships from the text as lines in the "
               "form 'EntityA | relationship | EntityB'. Output only those lines.\n\n"
               f"Text: {text}", max_tokens=200)
    triples = []
    for ln in out.splitlines():
        parts = [p.strip() for p in ln.split("|")]
        if len(parts) == 3 and all(parts):
            triples.append(tuple(parts))
    return triples

G = nx.Graph()
for doc in GRAPH_DOCS:
    for a, rel, b in extract_triples(doc):
        G.add_edge(a, b, relationship=rel)

print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
communities = [sorted(c) for c in greedy_modularity_communities(G)]
for i, c in enumerate(communities):
    print(f"community {i}:", c)

In [ ]:
def summarize_community(nodes):
    sub = G.subgraph(nodes)
    facts = [f"{a} {d.get('relationship', 'related to')} {b}" for a, b, d in sub.edges(data=True)]
    if not facts:
        return "(isolated nodes)"
    return chat("Write a 1 to 2 sentence summary of this community of facts:\n" +
                "\n".join(facts), max_tokens=120)

community_summaries = [summarize_community(c) for c in communities]
for i, s in enumerate(community_summaries):
    print(f"[community {i}] {s}\n")

In [ ]:
def global_search(query):
    # MAP: score each community summary for usefulness, keep the useful ones
    scored = []
    for summ in community_summaries:
        r = chat(f"Question: {query}\nSummary: {summ}\nHow useful is this summary for "
                 "answering the question, 0 to 10? Reply with a number only.", max_tokens=10)
        m = re.findall(r"\d+", r)
        scored.append((int(m[0]) if m else 0, summ))
    useful = [s for sc, s in sorted(scored, key=lambda x: -x[0]) if sc > 0][:3]
    # REDUCE: combine the kept summaries into one answer
    return chat("Using these community summaries, answer the question.\n\nSummaries:\n" +
                "\n".join(useful) + f"\n\nQuestion: {query}", max_tokens=250)

def local_search(query):
    seeds = [n for n in G.nodes if n.lower() in query.lower()]
    facts = []
    for n in seeds:
        for nb_ in G.neighbors(n):
            facts.append(f"{n} {G[n][nb_].get('relationship', 'related to')} {nb_}")
    if not facts:
        facts = [f"{a} {d.get('relationship')} {b}" for a, b, d in G.edges(data=True)]
    return chat("Answer the question using these facts.\n\nFacts:\n" + "\n".join(facts) +
                f"\n\nQuestion: {query}", max_tokens=200)

print("GLOBAL (sensemaking):")
print(global_search("What are the main themes connecting these events?"), "\n")
print("LOCAL (entity-centric):")
print(local_search("What happened to Rao?"))

## When to use which

| Pattern | Reach for it when |
|---|---|
| Adaptive RAG | question difficulty varies and latency matters |
| Self-RAG | grounding must be self-checked before shipping |
| CRAG | the store is patchy and freshness matters |
| Agentic RAG | multi-step lookups, multiple sources, tool use |
| GraphRAG | questions about relationships or the whole corpus |

They compose: an agent can route by complexity (Adaptive), grade and correct retrieval (CRAG), gate the answer on support (Self-RAG), and call a graph tool for relationship questions.

## What changes in production

| In this notebook | In production |
|---|---|
| plain-Python loops | LangGraph nodes with typed state and checkpointing |
| `web_search_stub` broadens retrieval | a real web tool (Tavily, AgentCore Web Search) |
| `greedy_modularity_communities` | the Leiden algorithm on the full corpus graph |
| access keys | an IAM role, least privilege, no hardcoded secrets |
| unbounded trust | step caps, per-call logging, retries and backoff, eval on a labeled set |